In [ ]:
from google.colab import files
uploaded = files.upload()

!unzip project.zip -d project
!mv project/project/* project/
!rm -r project/project

In [ ]:
import sys
sys.path.append("/content/project")

In [ ]:
RESUME = True   # Set to False if you want to start fresh
BEST = False    # Set to True to load the best checkpoint, False for last

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/transformer_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)

In [ ]:
def save_checkpoint(epoch, model, optimizer, val_loss, bleu, best=False):
    filename = "best.pt" if best else "last.pt"
    path = os.path.join(CHECKPOINT_DIR, filename)

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_loss": val_loss,
        "bleu": bleu,
    }, path)
    print(f"[Checkpoint] Saved {filename} at epoch {epoch+1}")


def load_checkpoint(model, optimizer, best=False):
    filename = "best.pt" if best else "last.pt"
    path = os.path.join(CHECKPOINT_DIR, filename)
    if os.path.exists(path):
        checkpoint = torch.load(path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])

        # Coerce BLEU to a float (avoid NoneType issues)
        bleu_val = checkpoint.get("bleu", 0.0) or 0.0
        bleu_str = f"{bleu_val:.2f}"

        print(f"[Checkpoint] Loaded {filename} from epoch {checkpoint['epoch']+1}, "
              f"Val Loss: {checkpoint['val_loss']:.4f}, BLEU: {bleu_str}")

        return checkpoint["epoch"], checkpoint["val_loss"], bleu_val
    else:
        print(f"[Checkpoint] No checkpoint found at {path}")
        return 0, float("inf"), 0.0


In [ ]:
# Install dependencies (Colab already has torch but let's ensure extras)
!pip install datasets sacrebleu transformers tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sacrebleu import corpus_bleu

from data import get_dataloader, tokenizer
from transformer import Transformer, make_src_mask, make_tgt_mask
from utils import translate_sentence

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SRC_VOCAB_SIZE = len(tokenizer)
TGT_VOCAB_SIZE = len(tokenizer)
PAD_IDX = tokenizer.pad_token_id

BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-4
MAX_LEN = 128

model = Transformer(
    src_vocab=SRC_VOCAB_SIZE,
    tgt_vocab=TGT_VOCAB_SIZE,
    d_model=512,    # larger model size
    N=6,            # deeper encoder/decoder
    heads=8,
    d_ff=2048,
    dropout=0.1,
    max_len=MAX_LEN
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
from transformers import AutoTokenizer

# Load the EN↔ES tokenizer fresh in the notebook
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")

# Ensure BOS/EOS exist (Helsinki models often have EOS but no BOS)
if tokenizer.bos_token is None:
    tokenizer.add_special_tokens({"bos_token": "<s>"})
if tokenizer.eos_token is None:
    tokenizer.add_special_tokens({"eos_token": "</s>"})

PAD_IDX = tokenizer.pad_token_id
BOS_IDX = tokenizer.bos_token_id
EOS_IDX = tokenizer.eos_token_id

print("Special token IDs ->",
      f"PAD: {PAD_IDX}",
      f"BOS: {BOS_IDX}",
      f"EOS: {EOS_IDX}",
      sep="\n")

# Use 128 for larger conversational datasets (opus100).
MAX_LEN = 128

In [ ]:
def train_epoch():
    model.train()

    dataloader = get_dataloader(split="train", batch_size=BATCH_SIZE, max_len=MAX_LEN)
    total_loss = 0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        src = batch["src"].to(DEVICE)
        tgt = batch["tgt"].to(DEVICE)

        # Teacher forcing: predict next token given all previous tokens
        tgt_input = tgt[:, :-1]
        tgt_labels = tgt[:, 1:]

        # Build masks for padding + future tokens
        src_mask = make_src_mask(src, PAD_IDX)
        tgt_mask = make_tgt_mask(tgt_input, PAD_IDX)

        # Forward pass
        logits = model(src, tgt_input, src_mask, tgt_mask)

        # Flatten for loss calculation
        logits = logits.reshape(-1, logits.size(-1))
        tgt_labels = tgt_labels.reshape(-1)

        loss = criterion(logits, tgt_labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
def evaluate(epoch=None):
    """
    Validation loop with BLEU scoring.
    - Always computes greedy BLEU (fast, runs every epoch).
    - Beam decoding is disabled here for speed; you can run it separately
      after training on a smaller validation subset.
    Returns:
        val_loss (float), bleu_greedy (float), bleu_beam (None)
    """
    model.eval()
    dataloader = get_dataloader(split="validation", batch_size=BATCH_SIZE//2, max_len=MAX_LEN)
    total_loss = 0
    all_preds_greedy, all_refs_greedy = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation", leave=False):
            src = batch["src"].to(DEVICE)
            tgt = batch["tgt"].to(DEVICE)

            # Shifted target for teacher forcing
            tgt_input = tgt[:, :-1]
            tgt_labels = tgt[:, 1:]

            # Build masks
            src_mask = make_src_mask(src, PAD_IDX)
            tgt_mask = make_tgt_mask(tgt_input, PAD_IDX)

            # Forward pass for validation loss
            logits = model(src, tgt_input, src_mask, tgt_mask)
            logits_reshaped = logits.reshape(-1, logits.size(-1))
            tgt_labels_reshaped = tgt_labels.reshape(-1)
            loss = criterion(logits_reshaped, tgt_labels_reshaped)
            total_loss += loss.item()

            # -------- GREEDY decoding --------
            preds = torch.argmax(logits, dim=-1)
            for p, r in zip(preds, tgt_labels):
                pred_ids = [id for id in p.tolist() if id not in [PAD_IDX, BOS_IDX, EOS_IDX]]
                ref_ids  = [id for id in r.tolist() if id not in [PAD_IDX, BOS_IDX, EOS_IDX]]
                pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True)
                ref_text  = tokenizer.decode(ref_ids, skip_special_tokens=True)

                all_preds_greedy.append(pred_text)
                all_refs_greedy.append([ref_text])

    # BLEU scores
    bleu_greedy = corpus_bleu(all_preds_greedy, all_refs_greedy).score
    bleu_beam = None   # disabled for training

    return total_loss / len(dataloader), bleu_greedy, bleu_beam

In [ ]:
start_epoch, best_val_loss, best_bleu = 0, float("inf"), 0.0

if RESUME:
    start_epoch, best_val_loss, best_bleu = load_checkpoint(model, optimizer, best=BEST)
else:
    print("[Training] Starting fresh (no checkpoint loaded)")

In [ ]:
train_losses, val_losses, bleus = [], [], []

for epoch in range(start_epoch, EPOCHS):
    # Training pass
    train_loss = train_epoch()

    # Validation pass (greedy always, beam optional)
    val_loss, bleu_greedy, bleu_beam = evaluate(epoch)

    # Save results
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    bleus.append({"greedy": bleu_greedy, "beam": bleu_beam})

    # Epoch summary
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss:   {train_loss:.4f}")
    print(f"  Val Loss:     {val_loss:.4f}")
    print(f"  BLEU (greedy): {bleu_greedy:.2f}")
    if bleu_beam is not None:
        print(f"  BLEU (beam):   {bleu_beam:.2f}")
    else:
        print("  BLEU (beam):   skipped during training")

    # Always save last checkpoint
    # If beam BLEU exists, we save that; otherwise greedy BLEU
    metric_for_checkpoint = bleu_beam if bleu_beam is not None else bleu_greedy
    save_checkpoint(epoch, model, optimizer, val_loss, metric_for_checkpoint, best=False)

    # Always use beam if available, otherwise greedy
    metric_for_checkpoint = bleu_beam if bleu_beam is not None else bleu_greedy

    # Track best checkpoint based on available metric
    if metric_for_checkpoint > best_bleu:
        best_bleu = metric_for_checkpoint
        save_checkpoint(epoch, model, optimizer, val_loss, metric_for_checkpoint, best=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, "last.pt"), map_location=DEVICE)

train_losses = checkpoint.get("train_losses", None)
val_losses   = checkpoint.get("val_losses", None)
bleus        = checkpoint.get("bleus", None)

if train_losses and val_losses:
    plt.figure(figsize=(10,5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.legend()
    plt.title("Loss Curves")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()

if bleus:
    bleu_greedy_scores = [b["greedy"] for b in bleus]
    bleu_beam_scores   = [b["beam"] if b["beam"] is not None else np.nan for b in bleus]

    plt.figure(figsize=(10,5))
    plt.plot(bleu_greedy_scores, label="BLEU (greedy)", marker="o")
    plt.plot(bleu_beam_scores, label="BLEU (beam)", marker="x")
    plt.legend()
    plt.title("Validation BLEU Scores")
    plt.xlabel("Epoch")
    plt.ylabel("BLEU")
    plt.show()
else:
    print("⚠️ No BLEU history saved in this checkpoint.")

In [ ]:
from utils import translate_sentence  # beam/greedy implementation from local files

def translate_sentence_colab(sentence, max_len=MAX_LEN, beam_size=10, use_beam=True):
    """
    Translate an input sentence using greedy or beam search decoding.

    Args:
        sentence (str): Input text (EN or ES).
        max_len (int): Max translation length.
        beam_size (int): Beam width (only used if use_beam=True).
        use_beam (bool): If True, runs beam search. If False, runs greedy decoding.

    Returns:
        str: Translated sentence.
    """
    if use_beam:
        print(f"[Demo] Translating with beam search (beam_size={beam_size})")
        return translate_sentence(sentence, model, max_len=max_len, beam_size=beam_size)
    else:
        print("[Demo] Translating with greedy decoding")
        return translate_sentence(sentence, model, max_len=max_len, beam_size=1)

In [ ]:
# Demo translations with both beam search and greedy decoding
test_sentences = [
    "How are you?",
    "I love learning machine translation.",
    "Buenos días, ¿cómo estás?",
    "La inteligencia artificial es fascinante."
]

for sent in test_sentences:
    print("="*60)
    print("Input:", sent)
    print("Beam (k=10):", translate_sentence_colab(sent, beam_size=10, use_beam=True))
    print("Greedy:", translate_sentence_colab(sent, use_beam=False))


In [ ]:
!jupyter nbconvert --ClearMetadataPreprocessor.enabled=True \
                   --to notebook \
                   --inplace your_notebook.ipynb